## Pratice for Context Management in OPENAI Agents SDK

In [11]:
from agents import Agent, Runner, function_tool
from agents import RunContextWrapper, TResponseInputItem
from dataclasses import dataclass
import random
import time

In [20]:
@dataclass
class UserProfile:
    id: int
    name: str
    shopping_cart: list[str]


@function_tool
def get_budget(wrapper: RunContextWrapper[UserProfile]):
    """Get the account Balance of the user using the user's id and thier linked bank account"""
    print("Getting Account Balance\n")
    time.sleep(0.5)
    user_id = wrapper.context.id

    # pretend we are fetching user's current balance from account
    return 100.0


@function_tool
def search_for_item(wrapper: RunContextWrapper[UserProfile], item: str) -> str:
    """Search for item in the database"""
    print("Searching the Item\n")
    time.sleep(0.5)
    # Randomly Generate the price for the item

    price = random.randint(1, 100)

    return f"found {item} in the database for price ${price}.00\n"


@function_tool
def get_shopping_cart(wrapper: RunContextWrapper[UserProfile]) -> str:
    """Gives names of item available in cart"""
    print("Getting Shopping Cart Items\n")
    time.sleep(0.5)
    return f"Items Selected :{", ".join(wrapper.context.shopping_cart)}\n"


@function_tool
def add_to_cart(wrapper: RunContextWrapper[UserProfile], item: str) -> None:
    """Adds the given item to the cart"""
    print(f"\n Adding {item} to cart\n")
    time.sleep(0.5)
    wrapper.context.shopping_cart.extend(item)


@function_tool
def purchase_item(wrapper: RunContextWrapper[UserProfile]) -> None:
    """gives names of the items Purchased from cart"""
    print("Purchasing Items in Cart\n")
    time.sleep(0.5)
    print(f"Items Purchased Successfully : {wrapper.context.shopping_cart}\n")

In [18]:
shopping_agent = Agent[UserProfile](
    name="Shopping Assistant",
    instructions="""You are a shopping assistant dedicated to helping the user with their grocery shopping needs.
    Your primary role is to assist in creating a shopping plan that fits within the user's budget.
    Start by getting the user's budget using the tool get_budget.
    Provide recommendations for grocery items based on the budget and user preferences.
    If the user is nearing or exceeding their budget, suggest cheaper alternatives or ask for a revised budget.
    If the user authorizes it, proceed with the purchase using the tool purchase_item.""",
    model="gpt-4o-mini",
    tools=[get_budget, search_for_item, get_shopping_cart, add_to_cart, purchase_item],
)

convo_items: list[TResponseInputItem] = []
print("You are chatting with the Shopping Assistant (type 'exit' to quit) ")
while True:

    user_input = input("You :")
    if "exit" in user_input:
        break

    profile1 = UserProfile(id=231, name="John Doe", shopping_cart=[])

    convo_items.append({"content": user_input, "role": "user"})

    result = await Runner.run(shopping_agent, convo_items, context=profile1)

    print(f"You : {user_input}")
    print(f"Shopping Assistant : {result.final_output}")
    convo_items = result.to_input_list()

You are chatting with the Shopping Assistant (type 'exit' to quit) 
You : hey
Shopping Assistant : Hello! How can I assist you today with your grocery shopping?
Getting Account Balance

You : can you check my current budget
Shopping Assistant : Your current budget is $100. How would you like to proceed with your grocery shopping? Do you have any specific items or preferences in mind?
Adding 1 liter milk to cart

You : add  1 ltr milk
Shopping Assistant : I've added 1 liter of milk to your cart. Would you like to add anything else or check your current cart?
Adding pack of bread to cart

You : now add a pack of bread
Shopping Assistant : I've added a pack of bread to your cart. Would you like to add more items or proceed to check out?
Getting Shopping Cart Items

You : how much is it
Shopping Assistant : It seems I can't retrieve the specific prices of the items in your cart at the moment. Would you like me to search for the prices of milk and bread so we can estimate the total?
Searchi